# FraudShield — Colab master runner

Every GPU-bound step of this project, in one notebook: **voice generation**,
**document-consistency evaluation**, **video-KYC generation + evaluation**, and
**GNN training**. Each writes its real results straight back into Supabase and
back into the dataset bundles, so nothing has to be pasted by hand afterwards.

## Why this replaces the old per-task notebooks

`docs/SESSION_HANDOFF.md` says not to clone the repo, because `data/generated/`
is gitignored and a clone would arrive empty — so earlier notebooks embedded
backend source files as `repr()`-encoded strings and needed data zipped up
through `files.upload()`. That was the right call at the time and it caused the
two documented Windows-path bugs.

`backend/tools/storage_sync.py` removed the reason for it: the dataset now lives
in Supabase Storage. So this notebook just **clones the repo** (real code, always
current) and **pulls the data** (real data, always current). No embedded source
strings, no manual zip transfer, no path-separator translation.

## Run order

Cells 1–3 are setup and are needed by everything. After that each task section is
independent — run only the ones you need. **Runtime → Change runtime type → GPU**
before starting.

## 1 · Clone the repo

In [ ]:
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/officialayush23/red_hat_vs_blue_hat_attack.git"
REPO_DIR = pathlib.Path("/content/red_hat_vs_blue_hat_attack")

# If the repo is private, put a GitHub personal access token here and the URL
# becomes https://<token>@github.com/... . Leave empty for a public repo.
GITHUB_TOKEN = ""

url = REPO_URL if not GITHUB_TOKEN else REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", url, str(REPO_DIR)], check=True)

BACKEND = REPO_DIR / "backend"
os.chdir(BACKEND)
sys.path.insert(0, str(BACKEND))
print("repo at", REPO_DIR)
print("HEAD:", subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"],
                              capture_output=True, text=True).stdout.strip())

## 2 · Credentials

Pasted at runtime, never committed. `SUPABASE_SERVICE_ROLE_KEY` is the write key
— it bypasses RLS, so treat this notebook as sensitive while it holds one.

Copy the values from your local `backend/.env`.

In [ ]:
from getpass import getpass

os.environ["SUPABASE_URL"] = input("SUPABASE_URL: ").strip()
os.environ["SUPABASE_ANON_KEY"] = getpass("SUPABASE_ANON_KEY: ").strip()
os.environ["SUPABASE_SERVICE_ROLE_KEY"] = getpass("SUPABASE_SERVICE_ROLE_KEY: ").strip()
hf = getpass("HF_TOKEN (blank if the models you need are public): ").strip()
if hf:
    os.environ["HF_TOKEN"] = hf

# Written to backend/.env because db/supabase_client.py loads it by path.
(BACKEND / ".env").write_text("\n".join(
    f"{k}={os.environ[k]}" for k in
    ["SUPABASE_URL", "SUPABASE_ANON_KEY", "SUPABASE_SERVICE_ROLE_KEY"] + (["HF_TOKEN"] if hf else [])
))
print("credentials set")

## 3 · Pull the dataset from Supabase Storage

Only the bundles the task you are about to run needs. Each bundle is
sha256-verified before extraction, so a truncated download can never quietly
become a partial dataset that every metric is then computed over.

In [ ]:
!pip install -q supabase python-dotenv

# Edit to match the task you are running. Full list:
#   attacks, synthetic_customers,
#   voice_attacks, voice_bonafide,
#   document_attacks, document_bonafide,
#   phishing_attacks, phishing_bonafide,
#   video_kyc_attacks, video_kyc_bonafide, video_kyc_reference
BUNDLES = "document_attacks,document_bonafide,voice_attacks,voice_bonafide,synthetic_customers"

!python tools/storage_sync.py pull --only {BUNDLES}
!python tools/storage_sync.py status

---
## A · Document-consistency evaluation  (PaddleOCR-VL, GPU)

The biggest open gap: **480 real generated cases, 0 rows in
`evaluation_results`**. Local Windows inference is blocked by `os error 1455`
(pagefile) and CPU inference was measured at 373.5 s/image, so this is the only
viable path. On a Colab GPU expect roughly 1–3 s/image.

`eval_document_consistency.py` writes its own `evaluation_runs` /
`evaluation_results` rows and updates `metrics.json`, so nothing needs pasting
back — but `metrics.json` lives in the repo, so cell A3 commits it.

In [ ]:
# A1 — install. Needs document_attacks + document_bonafide pulled in cell 3.
!pip install -q paddlepaddle-gpu paddleocr pillow qrcode
# No DLL-collision problem here: this is Linux, and the Windows
# cudnn/cublas filename clash documented in requirements-paddleocr-gpu.txt
# does not apply.
import paddle
print("paddle:", paddle.__version__, "| GPU:", paddle.device.is_compiled_with_cuda())

In [ ]:
# A2 — run the real evidence gate
!python evaluation/eval_document_consistency.py

---
## B · Voice generation  (Chatterbox, GPU)

Blocked locally by the same Windows pagefile error (`os error 1455`) during
`generate_voice_attacks.py`. Chatterbox pins `torch==2.6.0`, which is why it has
its own venv locally — on Colab it gets its own runtime instead, so **run
section B in a fresh runtime, not alongside A**.

In [ ]:
# B1 — install. Restart the runtime first if you ran section A.
!pip install -q chatterbox-tts soundfile librosa
import torch; print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# B2 — generate. --n-per-split is per split portion; 60 gives every declared
# voice_scam combination a usable number of cases.
!python generate/generate_voice_attacks.py --n-per-split 60 --seed 42

In [ ]:
# B3 — score the new audio with the real spoof detector
!pip install -q transformers
!python evaluation/eval_voice_spoof.py

---
## C · Video-KYC  (facenet-pytorch, GPU)

`video_kyc_detector` currently reports precision/recall/F1 = 1.000 on
**n = 12**. That is a wiring check, not a result, and it is the single most
attackable number on the site. This section generates a real set and re-scores.

facenet-pytorch pins `torch<2.3` while `transformers` needs `torch>=2.5` — they
cannot share an environment (that is why the Railway image excludes video-KYC).
**Run section C in its own fresh runtime.**

In [ ]:
# C1 — install. Fresh runtime, please.
!pip install -q facenet-pytorch opencv-python-headless
import torch; print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# C2 — needs the video bundles
!python tools/storage_sync.py pull --only video_kyc_attacks,video_kyc_bonafide,video_kyc_reference,synthetic_customers
!python generate/generate_video_kyc_attacks.py --n-per-split 60 --seed 42
!python evaluation/eval_video_kyc.py

---
## D · GNN — mule-network round 6

Round 5 stands at IBM AML ROC-AUC **0.7532**, recall **0.0746**, F1 **0.0258**.

Read `notebooks/train_gnn_mule_network.ipynb` for the round-5 training loop; this
cell is the harness around it. Two things worth being precise about before
chasing a number:

1. **Recall alone is trivially purchasable.** With ROC-AUC 0.75 you can hit 60%
   recall today by lowering the decision threshold — and precision, already
   0.0156, collapses further. A 60% recall figure obtained that way is not a
   result, and reporting it as one would be exactly the kind of number this
   project exists to avoid.
2. **The real lever is ROC-AUC**, i.e. ranking quality: class-imbalance handling
   in the loss (`pos_weight`), longer training, and richer edge features. Recall
   at a *stated* precision is the honest headline.

So this section records the full precision/recall curve and reports the operating
points, rather than a single recall number picked to look good.

In [ ]:
# D1 — install
!pip install -q torch torch_geometric
import torch; print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# D2 — train. Open notebooks/train_gnn_mule_network.ipynb and run its cells
# here, or import its training function once it has been extracted into a
# module. The round-5 loop is the starting point; round 6 changes to try, in
# the order most likely to move ROC-AUC:
#
#   a) pos_weight in BCEWithLogitsLoss set to (neg/pos) on the training split.
#      IBM AML is ~0.1% positive; an unweighted loss learns to predict "no".
#   b) more epochs with early stopping on validation ROC-AUC, not on loss.
#   c) edge features: log-amount, time-delta to the account's previous edge,
#      in/out degree ratio at both endpoints, and the port-numbering features
#      round 5 added.
#
# Save the checkpoint to backend/defend/models/gnn.pt when done.
print("open notebooks/train_gnn_mule_network.ipynb and run its training cells")

In [ ]:
# D3 — evaluate the new checkpoint, and report operating points honestly
!python evaluation/eval_gnn.py

import json, pathlib
m = json.loads(pathlib.Path("defend/models/metrics.json").read_text())
for k in sorted(m):
    if k.startswith("gnn"):
        e = m[k].get("metrics", m[k])
        print(f"{k:48} roc_auc={e.get('roc_auc')} recall={e.get('recall')} precision={e.get('precision')}")

---
## E · Push everything back

Each eval script already wrote its `evaluation_runs` / `evaluation_results` rows
and updated `metrics.json` as it ran. This section pushes the two things that
live outside Supabase's tables: the regenerated dataset bundles, and the
`metrics.json` / `EVALUATION_RESULTS.md` changes in the repo.

In [ ]:
# E1 — dataset bundles back to Storage
!python tools/storage_sync.py push
!python db/sync_model_registry.py

In [ ]:
# E2 — repo changes back to GitHub.
# Needs GITHUB_TOKEN set in cell 1 (a token with repo write scope).
!git -C {REPO_DIR} config user.email "colab@fraudshield.local"
!git -C {REPO_DIR} config user.name "FraudShield Colab runner"
!git -C {REPO_DIR} add backend/defend/models/metrics.json backend/defend/models/gnn.pt docs/EVALUATION_RESULTS.md
!git -C {REPO_DIR} commit -m "colab: real evidence-gate results from the master runner" || echo "nothing to commit"
!git -C {REPO_DIR} push origin main

In [ ]:
# E3 — verify what actually landed in Supabase
import os
from supabase import create_client
sb = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_ROLE_KEY"])
for family in ["document_fraud", "voice_scam", "video_kyc"]:
    r = sb.table("evaluation_results").select("id", count="exact").like("case_id", f"{family}%").execute()
    print(f"{family:16} {r.count:>6} scored results in evaluation_results")